In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. Read raw data
metrics = pd.read_csv("metrics.csv")
vendor = pd.read_csv("Vendor.csv")

In [3]:
# 2. Check column names
print("Metrics columns:")
print(metrics.columns)

print("\nVendor columns:")
print(vendor.columns)


Metrics columns:
Index(['Date', 'Sub Category ID', 'Plant ID', 'Vendor ID', 'Material ID',
       'Defect Type ID', 'Material Type ID', 'Defect ID', 'Defect Qty',
       'Downtime min'],
      dtype='object')

Vendor columns:
Index(['Vendor', 'Vendor ID'], dtype='object')


In [4]:
# 3. Aggregate supplier-level metrics
supplier_summary = (
    metrics
    .groupby("Vendor ID")
    .agg(
        total_defect_qty=("Defect Qty", "sum"),
        total_downtime_minutes=("Downtime min", "sum"),
        total_defect_reports=("Defect Qty", "count")
    )
    .reset_index()
)

In [5]:
supplier_summary = supplier_summary.merge(
    vendor,
    on="Vendor ID",
    how="left"
)
print(supplier_summary)

     Vendor ID  total_defect_qty  total_downtime_minutes  \
0            1            484594                   26185   
1            2           3838867                   10405   
2            3             36658                       0   
3            4           3169402                    5978   
4            5             72783                     720   
..         ...               ...                     ...   
315        324                 0                       0   
316        325                23                       0   
317        326                 0                       0   
318        327              1717                       0   
319        328             34390                       0   

     total_defect_reports       Vendor  
0                     710      Reddoit  
1                     323      Plustax  
2                      10       bamity  
3                     396    Quotelane  
4                      16       Viatom  
..                    ...        

In [6]:
def normalize(series):
    """
    Convert a numeric column into a 0-100 score.
    The highest value becomes 100, and the lowest value becomes 0.
    """
    if series.max() == series.min():
        return pd.Series(0, index=series.index)
    
    return (series - series.min()) / (series.max() - series.min()) * 100

In [7]:
supplier_summary["downtime_risk_score"] = normalize(
    supplier_summary["total_downtime_minutes"]
)

supplier_summary["defect_qty_risk_score"] = normalize(
    supplier_summary["total_defect_qty"]
)

supplier_summary["report_frequency_risk_score"] = normalize(
    supplier_summary["total_defect_reports"]
)

In [8]:
supplier_summary["supplier_risk_score"] = (
    supplier_summary["downtime_risk_score"] * 0.40
    + supplier_summary["defect_qty_risk_score"] * 0.35
    + supplier_summary["report_frequency_risk_score"] * 0.25
)

In [9]:
supplier_summary["supplier_risk_score"] = supplier_summary["supplier_risk_score"].round(2)

In [10]:
def assign_risk_level(score):
    if score >= 70:
        return "High Risk"
    elif score >= 40:
        return "Medium Risk"
    else:
        return "Low Risk"

supplier_summary["risk_level"] = supplier_summary["supplier_risk_score"].apply(assign_risk_level)

In [11]:
def generate_recommendation(row):
    if row["risk_level"] == "High Risk":
        return "Prioritize supplier review and corrective action plan."
    elif row["risk_level"] == "Medium Risk":
        return "Monitor supplier performance and investigate recurring issues."
    else:
        return "Maintain regular monitoring."

supplier_summary["recommendation"] = supplier_summary.apply(generate_recommendation, axis=1)

In [12]:
supplier_summary = supplier_summary.sort_values(
    by="supplier_risk_score",
    ascending=False
)

In [14]:
supplier_summary.to_csv("supplier_risk_summary.csv", index=False)
print("supplier_risk_summary.csv has been created successfully.")
print(supplier_summary.head(10))

supplier_risk_summary.csv has been created successfully.
    Vendor ID  total_defect_qty  total_downtime_minutes  total_defect_reports  \
0           1            484594                   26185                   710   
1           2           3838867                   10405                   323   
3           4           3169402                    5978                   396   
34         36           3988201                    2300                   118   
10         11           2982348                    3088                   187   
8           9            735527                   10006                   404   
19         20           2589319                    4215                    96   
15         16           2005374                    3327                   152   
56         62            513653                   10275                   170   
13         14           1274282                    2221                   255   

         Vendor  downtime_risk_score  defect_qty_ri